# D164 — Introduction to MySQL Window Functions

A window function calculates across a set of related rows while keeping every original result row visible.

A normal `GROUP BY` reduces many rows to one row per group. A window function does not collapse those rows. This makes window functions useful when a detail row must appear beside a total, average, previous value, running result, or group comparison.

## Common use cases

Window functions are commonly used for:

- showing each sale beside the employee or region total;
- calculating running totals;
- smoothing data with moving averages;
- comparing a row with the previous or next row;
- calculating change over time;
- finding the first or last value in a group;
- calculating each row's percentage contribution; and
- ranking and distribution analysis, introduced in D165–D167.

## 1. Connect and create ten simple records

In [ ]:
import os
import mysql.connector
connection=mysql.connector.connect(
 host=os.environ.get('MYSQL_HOSTNAME','127.0.0.1'),
 port=int(os.environ.get('MYSQL_PORT','3306')),
 user=os.environ.get('MYSQL_USERNAME','root'),
 password=os.environ.get('MYSQL_PASSWORD','root'),
 database=os.environ.get('MYSQL_DATABASE','olist_import_lab'))
print('Connected:',connection.is_connected())

In [ ]:
def execute_sql(sql):
 cursor=connection.cursor(); cursor.execute(sql)
 if not cursor.with_rows:
  connection.commit(); print(f'Affected rows: {cursor.rowcount}'); cursor.close(); return []
 columns=[x[0] for x in cursor.description]; rows=cursor.fetchall(); cursor.close()
 text=[['NULL' if v is None else str(v) for v in row] for row in rows]
 widths=[len(c) for c in columns]
 for row in text: widths=[max(w,len(v)) for w,v in zip(widths,row)]
 print(' | '.join(c.ljust(w) for c,w in zip(columns,widths)))
 print('-+-'.join('-'*w for w in widths))
 for row in text: print(' | '.join(v.ljust(w) for v,w in zip(row,widths)))
 return rows

In [ ]:
execute_sql('DROP TEMPORARY TABLE IF EXISTS window_demo_sales')
execute_sql("""CREATE TEMPORARY TABLE window_demo_sales(
 sale_id INT PRIMARY KEY, employee VARCHAR(20), region VARCHAR(10),
 sale_date DATE, amount DECIMAL(10,2))""")
execute_sql("""INSERT INTO window_demo_sales VALUES
 (1,'Asha','South','2025-01-05',1200),(2,'Asha','South','2025-01-18',800),
 (3,'Bilal','North','2025-01-07',1500),(4,'Bilal','North','2025-02-04',1800),
 (5,'Chen','South','2025-01-22',700),(6,'Chen','South','2025-02-11',900),
 (7,'Divya','North','2025-02-15',2100),(8,'Divya','North','2025-03-02',1100),
 (9,'Eshan','South','2025-03-08',600),(10,'Eshan','South','2025-03-20',1000)""")
execute_sql('SELECT * FROM window_demo_sales ORDER BY sale_date,sale_id')

## 2. `GROUP BY` compared with a window

`GROUP BY employee` produces one row per employee. Individual sales are no longer visible.

In [ ]:
execute_sql("""SELECT employee,COUNT(*) sale_count,SUM(amount) employee_total
FROM window_demo_sales GROUP BY employee ORDER BY employee""")

Adding `OVER(PARTITION BY employee)` changes `SUM` into a window function. All ten sales remain visible, and the employee total repeats beside the employee's rows.

A **partition** is a group of rows used by a window calculation. It does not collapse the rows.

In [ ]:
execute_sql("""SELECT sale_id,employee,amount,
 SUM(amount) OVER(PARTITION BY employee) employee_total
FROM window_demo_sales ORDER BY employee,sale_date""")

## 3. Empty `OVER()` means all result rows

`OVER()` has no partition or window ordering. Every row belongs to one large window. This example displays each sale beside the grand total and overall average.

In [ ]:
execute_sql("""SELECT sale_id,employee,amount,
 SUM(amount) OVER() all_sales, ROUND(AVG(amount) OVER(),2) average_sale,
 MIN(amount) OVER() smallest_sale,MAX(amount) OVER() largest_sale
FROM window_demo_sales ORDER BY sale_id""")

## 4. Use several partitions

A query can calculate different windows at the same time. Here, each row shows employee, region, and overall totals.

In [ ]:
execute_sql("""SELECT sale_id,employee,region,amount,
 SUM(amount) OVER(PARTITION BY employee) employee_total,
 SUM(amount) OVER(PARTITION BY region) region_total,
 SUM(amount) OVER() grand_total
FROM window_demo_sales ORDER BY region,employee,sale_date""")

## 5. Percentage contribution

A row's amount divided by a window total gives its contribution. `NULLIF(total,0)` prevents division by zero.

In [ ]:
execute_sql("""SELECT sale_id,employee,region,amount,
 ROUND(100*amount/NULLIF(SUM(amount) OVER(PARTITION BY region),0),2) region_percent,
 ROUND(100*amount/NULLIF(SUM(amount) OVER(),0),2) all_sales_percent
FROM window_demo_sales ORDER BY region,sale_date""")

## 6. Window ordering creates a sequence

`ORDER BY` inside `OVER` defines the calculation sequence. It is separate from the final `ORDER BY`, which controls how results are displayed.

The running total includes the current row and every earlier row in date order. `sale_id` is added as a tie-break in case two sales have the same date.

In [ ]:
execute_sql("""SELECT sale_id,sale_date,employee,amount,
 SUM(amount) OVER(ORDER BY sale_date,sale_id
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) running_total
FROM window_demo_sales ORDER BY sale_date,sale_id""")

## 7. Window frames

A **frame** is the subset of ordered window rows used for the current calculation.

- `UNBOUNDED PRECEDING` means the first row in the partition.
- `2 PRECEDING` means two rows before the current row.
- `CURRENT ROW` means the current row.
- `UNBOUNDED FOLLOWING` means the final row in the partition.

Use an explicit `ROWS` frame when the intended row range matters.

## 8. Moving average

A moving average reduces short-term variation. This three-row average includes the current sale and two earlier sales. The first rows use fewer than three values because fewer earlier rows exist.

In [ ]:
execute_sql("""SELECT sale_id,sale_date,amount,
 ROUND(AVG(amount) OVER(ORDER BY sale_date,sale_id
  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),2) three_sale_average
FROM window_demo_sales ORDER BY sale_date,sale_id""")

## 9. `LAG`: look at an earlier row

`LAG(amount)` returns the amount from the previous row in the window order. It is useful for period-over-period change. The first row has no previous value, so it returns `NULL`.

In [ ]:
execute_sql("""WITH compared AS(
 SELECT sale_id,sale_date,amount,
  LAG(amount) OVER(ORDER BY sale_date,sale_id) previous_amount
 FROM window_demo_sales)
SELECT *,amount-previous_amount change_from_previous
FROM compared ORDER BY sale_date,sale_id""")

## 10. `LEAD`: look at a later row

`LEAD` returns a value from a following row. Here, it shows each employee's next sale date and the days until that sale. `PARTITION BY employee` prevents one employee from seeing another employee's date.

In [ ]:
execute_sql("""WITH next_sales AS(
 SELECT sale_id,employee,sale_date,
  LEAD(sale_date) OVER(PARTITION BY employee ORDER BY sale_date,sale_id) next_sale_date
 FROM window_demo_sales)
SELECT *,DATEDIFF(next_sale_date,sale_date) days_until_next_sale
FROM next_sales ORDER BY employee,sale_date""")

## 11. `FIRST_VALUE` and `LAST_VALUE`

`FIRST_VALUE` returns the first value in the ordered window. `LAST_VALUE` needs care: its default frame often ends at the current row. To obtain the final value in the whole employee partition, the frame below ends with `UNBOUNDED FOLLOWING`.

In [ ]:
execute_sql("""SELECT employee,sale_date,amount,
 FIRST_VALUE(amount) OVER(PARTITION BY employee ORDER BY sale_date,sale_id) first_amount,
 LAST_VALUE(amount) OVER(PARTITION BY employee ORDER BY sale_date,sale_id
  ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) last_amount
FROM window_demo_sales ORDER BY employee,sale_date""")

## 12. Filter a window result through a CTE

Window functions are calculated after `WHERE`. Therefore, an alias such as `employee_average` cannot be filtered in the same query level. Calculate it in a CTE, then filter in the outer query.

In [ ]:
execute_sql("""WITH compared AS(
 SELECT *,AVG(amount) OVER(PARTITION BY employee) employee_average
 FROM window_demo_sales)
SELECT sale_id,employee,amount,ROUND(employee_average,2) employee_average
FROM compared WHERE amount>employee_average
ORDER BY employee,amount DESC""")

## 13. Named windows

When several functions use the same window definition, MySQL's `WINDOW` clause can name it once. This reduces repetition.

In [ ]:
execute_sql("""SELECT sale_id,employee,amount,
 SUM(amount) OVER employee_window employee_total,
 ROUND(AVG(amount) OVER employee_window,2) employee_average,
 COUNT(*) OVER employee_window employee_sale_count
FROM window_demo_sales
WINDOW employee_window AS(PARTITION BY employee)
ORDER BY employee,sale_date""")

## 14. Window-function checklist

Before writing a window function, answer four questions:

1. What does one result row represent?
2. Should the calculation use all rows or restart by a partition?
3. Does the calculation require an order?
4. If ordered, which frame should be used?

Use `GROUP BY` when only one result row per group is needed. Use a window function when detail rows must remain visible beside group or sequence calculations. D169 applies these ideas to the Olist order-items table.

In [ ]:
connection.close()
print('MySQL connection closed; the temporary table is gone.')